In [1]:
import numpy as np
import pandas as pd

In [2]:
import os

path = "../../data/processed/"
dfs = {}

# read 03_CLEAN_COMPLETE_DF.parquet
sites = pd.read_parquet(os.path.join(path, "dep_codes.parquet"))


In [3]:
pd.set_option('display.max_columns', None)

In [4]:


sites_names = sites[['HERlvl1Code', 'HERlvl1Name']]

# Mantener solo los valores únicos de 'col1'
sites_names = sites_names.drop_duplicates(subset='HERlvl1Code')

In [5]:
ranges = pd.read_csv('ranges.csv')
ranges

,HERlvl1Name,IBD_EQR_Status,IBD_min,IBD_max,IBD_mid,HERlvl1Code
0,ALPES INTERNES,Bad,0.000,9.800,9.30,2
1,ALPES INTERNES,Poor,9.800,13.225,10.30,2
2,ALPES INTERNES,Moderate,13.225,17.025,16.15,2
3,ALPES INTERNES,Good,17.025,18.725,17.90,2
4,ALPES INTERNES,High,18.725,20.000,19.55,2
...,...,...,...,...,...,...
105,VOSGES,Bad,0.000,9.550,7.90,4
106,VOSGES,Poor,9.550,12.775,11.20,4
107,VOSGES,Moderate,12.775,15.700,14.35,4
108,VOSGES,Good,15.700,18.075,17.05,4


In [8]:
t = pd.read_csv('metricts_t.csv')
epm = pd.read_csv('metricts_epm.csv')
dirty = pd.read_csv('metricts_dirty.csv')
tuneado = pd.read_csv('metrics_tuneado.csv')


In [9]:
t

,region,R2_train,R2_valid,MAE_train,MAE_valid,RMSE_train,RMSE_valid,best_iterations
0,18,0.999984,0.901764,0.007065,0.475877,6.880693e-05,0.539792,2991
1,5,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,2997
2,4,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,2196
3,10,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,2993
4,22,1.000000,0.562962,0.000040,1.190931,1.913874e-09,3.673909,1652
5,9,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,2985
6,21,0.999565,0.928841,0.045767,0.463063,3.003642e-03,0.525956,2999
7,20,1.000000,0.867077,0.000260,0.726611,8.974746e-08,0.842552,2980
8,12,0.997816,0.944985,0.088146,0.353910,1.179671e-02,0.311098,2995
9,8,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,1482


In [10]:
import pandas as pd
from functools import reduce

# --- 1) gap = |R2_train - R2_valid| en cada df ---
def add_gap(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # asegúrate de que sean numéricos por si vienen como str
    out["R2_train"] = pd.to_numeric(out["R2_train"], errors="coerce")
    out["R2_valid"] = pd.to_numeric(out["R2_valid"], errors="coerce")
    out["gap"] = (out["R2_train"] - out["R2_valid"]).abs()
    return out

t_gap     = add_gap(t)
epm_gap   = add_gap(epm)
dirty_gap = add_gap(dirty)
tun_gap   = add_gap(tuneado)

# --- 2) renombrar todas las columnas excepto el id ('region') con sufijos t, epm, dirty ---
def with_suffix(df: pd.DataFrame, suf: str) -> pd.DataFrame:
    rename_map = {c: f"{c}_{suf}" for c in df.columns if c != "region"}
    return df.rename(columns=rename_map)

t_s     = with_suffix(t_gap, "t")
epm_s   = with_suffix(epm_gap, "epm")
dirty_s = with_suffix(dirty_gap, "dirty")
tun_s   =  with_suffix(tun_gap, "tun" )

# --- 3) merge de los 3 por 'region' ---
merged = reduce(lambda l, r: pd.merge(l, r, on="region", how="outer"), [t_s, epm_s, dirty_s, tun_s])

# --- 4) columnas resumen: quién tiene menor gap, mayor R2_train y mayor R2_valid ---
# (maneja empates concatenando con '+')
def winners(df: pd.DataFrame, cols_prefix: str, strip_prefix: str, how="min") -> pd.Series:
    cols = [c for c in df.columns if c.startswith(cols_prefix)]
    mat = df[cols]
    if how == "min":
        flags = mat.eq(mat.min(axis=1), axis=0)
    else:
        flags = mat.eq(mat.max(axis=1), axis=0)
    return flags.apply(lambda row: "+".join(row.index[row].str.replace(strip_prefix, "")), axis=1)

# menor gap
merged["min_gap_from"] = winners(merged, "gap_", "gap_")

# mayor R2_train
merged["max_R2_train_from"] = winners(merged, "R2_train_", "R2_train_", how="max")

# mayor R2_valid
merged["max_R2_valid_from"] = winners(merged, "R2_valid_", "R2_valid_", how="max")

# (opcional) reordenar para ver primero 'region' y los 3 resúmenes
summary_cols = ["region", "min_gap_from", "max_R2_train_from", "max_R2_valid_from"]
other_cols = [c for c in merged.columns if c not in summary_cols]
merged = merged[summary_cols + other_cols]




In [11]:
merged

,region,min_gap_from,max_R2_train_from,max_R2_valid_from,R2_train_t,R2_valid_t,MAE_train_t,MAE_valid_t,RMSE_train_t,RMSE_valid_t,best_iterations_t,gap_t,R2_train_epm,R2_valid_epm,MAE_train_epm,MAE_valid_epm,RMSE_train_epm,RMSE_valid_epm,best_iterations_epm,gap_epm,R2_train_dirty,R2_valid_dirty,MAE_train_dirty,MAE_valid_dirty,RMSE_train_dirty,RMSE_valid_dirty,best_iterations_dirty,gap_dirty,R2_train_tun,R2_valid_tun,MAE_train_tun,MAE_valid_tun,RMSE_train_tun,RMSE_valid_tun,best_iterations_tun,gap_tun,used_ohe_tun
0,1,tun,dirty,tun,0.999732,0.771942,0.019290,0.337553,5.078926e-04,0.502332,2999,0.227790,0.999802,0.715876,0.016048,0.378590,3.743185e-04,0.625826,2563,0.283926,0.999966,0.762488,0.006728,0.361597,6.388313e-05,0.523157,2999,0.237479,0.986495,0.814072,0.080366,0.301853,0.025556,0.409534,4971,0.172422,False
1,2,tun,t,tun,0.999985,0.700933,0.004353,0.196092,2.528543e-05,0.279299,763,0.299052,0.999342,0.586681,0.026409,0.243060,1.110844e-03,0.385999,408,0.412661,0.999316,0.678723,0.025303,0.211337,1.154054e-03,0.300041,301,0.320593,0.993755,0.725479,0.031217,0.194268,0.010541,0.256375,4947,0.268275,False
2,3,tun,dirty,tun,0.996929,0.958966,0.122453,0.396599,2.261249e-02,0.314263,2999,0.037962,0.991239,0.903985,0.203539,0.603054,6.450311e-02,0.735342,2941,0.087254,0.998467,0.957690,0.086528,0.400418,1.129034e-02,0.324035,2999,0.040776,0.990862,0.962511,0.191459,0.378320,0.067281,0.287116,4995,0.028351,False
3,4,tun,epm,tun,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,2196,0.166717,1.000000,0.821916,0.000034,0.756563,1.572097e-09,1.370564,2991,0.178084,1.000000,0.816799,0.000103,0.816917,1.472407e-08,1.409947,2641,0.183201,0.998325,0.853600,0.080948,0.668595,0.012130,1.126720,4993,0.144725,False
4,5,tun,dirty,tun,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,2997,0.069960,0.999656,0.903486,0.038122,0.512000,2.132275e-03,0.692717,2930,0.096170,0.999879,0.927646,0.022638,0.448746,7.502270e-04,0.519312,2996,0.072233,0.995395,0.945440,0.121711,0.377442,0.028542,0.391596,4999,0.049954,False
5,6,tun,dirty,tun,0.999606,0.945793,0.053201,0.502085,4.042070e-03,0.605457,2997,0.053813,0.999601,0.892819,0.052850,0.731376,4.093465e-03,1.197141,2967,0.106782,0.999874,0.933357,0.029776,0.558042,1.287581e-03,0.744360,2951,0.066518,0.994405,0.952460,0.179781,0.486937,0.057355,0.530994,4999,0.041945,False
6,7,tun,t,t,0.999996,0.910870,0.003824,0.261666,1.947808e-05,0.340529,1770,0.089126,0.998796,0.859979,0.057239,0.325643,5.393189e-03,0.534964,472,0.138818,0.999899,0.878960,0.017750,0.284930,4.508090e-04,0.462445,769,0.120940,0.995127,0.910155,0.073230,0.298223,0.021831,0.343261,4932,0.084972,False
7,8,t,dirty,t,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,1482,0.157427,1.000000,0.759560,0.000500,0.679163,3.413241e-07,1.047408,2387,0.240440,1.000000,0.769914,0.000150,0.673607,3.010695e-08,1.002304,2906,0.230086,0.991100,0.829515,0.121444,0.614152,0.037314,0.742670,4689,0.161585,False
8,9,tun,dirty,tun,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,2985,0.047801,0.969989,0.855824,0.237858,0.419919,9.886818e-02,0.466024,2999,0.114165,0.992972,0.939320,0.119549,0.270092,2.315491e-02,0.196139,2996,0.053652,0.973284,0.944718,0.190055,0.256473,0.088016,0.178690,4998,0.028566,False
9,10,tun,dirty,tun,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,2993,0.049168,0.998449,0.914799,0.092073,0.515502,1.274664e-02,0.685365,2984,0.083650,0.999443,0.943318,0.055295,0.415377,4.575315e-03,0.455956,2999,0.056125,0.988242,0.950346,0.199624,0.391176,0.096613,0.399420,4999,0.037896,False


In [13]:
def choose_best_balanced(row, gap_threshold=0.15):
    # 1. modelo con mejor R²_valid
    best_r2 = row["max_R2_valid_from"]
    
    # 2. si su gap es alto, considerar al más estable
    gap_value = row[f"gap_{best_r2}"]
    if gap_value > gap_threshold:
        return row["min_gap_from"]  # cambio por estabilidad
    else:
        return best_r2

merged["best_final_model"] = merged.apply(choose_best_balanced, axis=1)

merged[["best_final_model",'region' ]]

,best_final_model,region
0,tun,1
1,tun,2
2,tun,3
3,tun,4
4,tun,5
5,tun,6
6,t,7
7,t,8
8,tun,9
9,tun,10


In [19]:
merged[["best_final_model",'region', 'R2_train_t', 'R2_valid_t', 'R2_train_tun', 'R2_valid_tun' , 'R2_train_dirty', 'R2_valid_dirty' ]]

,best_final_model,region,R2_train_t,R2_valid_t,R2_train_tun,R2_valid_tun,R2_train_dirty,R2_valid_dirty
0,tun,1,0.999732,0.771942,0.986495,0.814072,0.999966,0.762488
1,tun,2,0.999985,0.700933,0.993755,0.725479,0.999316,0.678723
2,tun,3,0.996929,0.958966,0.990862,0.962511,0.998467,0.957690
3,tun,4,0.999998,0.833281,0.998325,0.853600,1.000000,0.816799
4,tun,5,0.999617,0.929657,0.995395,0.945440,0.999879,0.927646
5,tun,6,0.999606,0.945793,0.994405,0.952460,0.999874,0.933357
6,t,7,0.999996,0.910870,0.995127,0.910155,0.999899,0.878960
7,t,8,0.999931,0.842503,0.991100,0.829515,1.000000,0.769914
8,tun,9,0.989169,0.941368,0.973284,0.944718,0.992972,0.939320
9,tun,10,0.998872,0.949704,0.988242,0.950346,0.999443,0.943318


In [29]:
t_p = pd.read_csv('predicciones_t.csv')
epm_p = pd.read_csv('predicciones_epm.csv')
dirty_p = pd.read_csv('predicciones_dirty.csv')
preds = pd.read_csv('preds.csv')

In [30]:
preds

,SamplingOperations_code,IBD_pred,region
0,S02000010_20080811,15.062039,18
1,S02000010_20100719,15.438521,18
2,S02000010_20150811,13.819270,18
3,S02000010_20160825,15.086803,18
4,S02000010_20170703,16.036757,18
...,...,...,...
5658,S06700075_20120611,20.008280,2
5659,S06700094_20210830,18.611227,2
5660,S06700590_20210830,18.366359,2
5661,S06710014_20210823,19.538874,2


In [31]:
con_nombre = pd.merge(preds,sites_names,left_on='region',right_on='HERlvl1Code', how = 'left')

In [32]:
con_nombre

,SamplingOperations_code,IBD_pred,region,HERlvl1Code,HERlvl1Name
0,S02000010_20080811,15.062039,18,18,ALSACE
1,S02000010_20100719,15.438521,18,18,ALSACE
2,S02000010_20150811,13.819270,18,18,ALSACE
3,S02000010_20160825,15.086803,18,18,ALSACE
4,S02000010_20170703,16.036757,18,18,ALSACE
...,...,...,...,...,...
5658,S06700075_20120611,20.008280,2,2,ALPES INTERNES
5659,S06700094_20210830,18.611227,2,2,ALPES INTERNES
5660,S06700590_20210830,18.366359,2,2,ALPES INTERNES
5661,S06710014_20210823,19.538874,2,2,ALPES INTERNES


In [33]:
def to_status(yhat: pd.DataFrame, ranges: pd.DataFrame) -> pd.DataFrame:
    """
    Adds the column 'IBD_EQR_Status_Predicted' to the `yhat` DataFrame by mapping the predicted IBD values 
    ('IBD_Predicted') into the appropriate bin defined by the [IBD_min, IBD_max) intervals in the `ranges` DataFrame 
    for the corresponding 'HERlvl1Name'. The topmost bin per region also includes its right endpoint.

    Parameters:
    ----------
    yhat : pd.DataFrame
        A DataFrame containing the predicted IBD values ('IBD_Predicted') and the corresponding 'HERlvl1Name'.
    ranges : pd.DataFrame
        A DataFrame containing the bin definitions for each 'HERlvl1Name', including columns:
        - 'HERlvl1Name': The region name.
        - 'IBD_EQR_Status': The status corresponding to the bin.
        - 'IBD_min': The lower bound of the bin (inclusive).
        - 'IBD_max': The upper bound of the bin (exclusive, except for the topmost bin).

    Returns:
    -------
    pd.DataFrame
        A copy of the `yhat` DataFrame with an additional column 'IBD_EQR_Status_Predicted', which contains the 
        mapped status for each prediction.

    Notes:
    -----
    - The function performs a cartesian merge between `yhat` and `ranges` based on 'HERlvl1Name'.
    - Each predicted value is matched to the bin where it falls within the [IBD_min, IBD_max) interval.
    - For the topmost bin in each region, the right endpoint (IBD_max) is included.
    - In case of ties (multiple bins matching a prediction), the first match is kept.
    """
    out = yhat.copy()
    out['__ix__'] = np.arange(len(out))

    # Copy ranges and calculate the maximum right endpoint for each region
    r = ranges[['HERlvl1Name', 'IBD_EQR_Status', 'IBD_min', 'IBD_max']].copy()
    r['__max_right__'] = r.groupby('HERlvl1Name')['IBD_max'].transform('max')

    # Cartesian merge by region, then keep the single interval that matches each prediction
    m = out.merge(r, on='HERlvl1Name', how='left')

    # Check if predictions fall within the bin intervals
    pred = m['IBD_pred'].astype(float)
    left_ok  = pred >= m['IBD_min']
    right_ok = (pred <  m['IBD_max']) | ((pred == m['IBD_max']) & (m['IBD_max'].eq(m['__max_right__'])))
    m = m[left_ok & right_ok]

    # In case of any ties, keep the first match; then map back to original rows
    m = m.sort_values(['__ix__', 'IBD_min', 'IBD_max']).drop_duplicates('__ix__', keep='first')
    status = m.set_index('__ix__')['IBD_EQR_Status']

    # Map the status back to the original DataFrame
    out['IBD_EQR_Status_Predicted'] = out['__ix__'].map(status)
    out = out.drop(columns='__ix__')
    return out

def get_results(yhat, ranges, output_file="IBD_EQR_Status_predictions_5663.csv") -> pd.Series:
    yhat2 = to_status(yhat, ranges)
    send_predictions = yhat2['IBD_EQR_Status_Predicted']
    send_predictions = send_predictions.to_frame()
    send_predictions = send_predictions.rename(columns={'IBD_EQR_Status_Predicted': 'IBD_EQR_Status'})
    send_predictions.to_csv(output_file, index=True)
    print(f"Saved predictions to {output_file}")
    return send_predictions

def process_predictions(yhat: pd.DataFrame, ranges: pd.DataFrame, output_file="IBD_EQR_Status_predictions.csv") -> pd.Series:
    """
    Processes predictions by adding EQR status, comparing with an example dataset, and saving the results.

    Parameters:
    ----------
    yhat : pd.DataFrame
        DataFrame containing predicted IBD values and corresponding HERlvl1Name.
    ranges : pd.DataFrame
        DataFrame containing bin definitions for mapping IBD values to EQR status.
    read_example : pd.DataFrame
        DataFrame containing example predictions for comparison.
    output_file : str
        Path to save the processed predictions as a CSV file.

    Returns:
    -------
    pd.Series
        Processed predictions with the name 'IBD_EQR_Status'.
    """

    read_example = pd.read_csv("..\\..\\results\\000 example\\random_predictions.csv")
    # Add EQR status to predictions
    yhat2 = to_status(yhat, ranges)
    send_predictions = yhat2['IBD_EQR_Status_Predicted']

    # Inner join to compare with example predictions
    innerjoin = send_predictions.to_frame().merge(
        read_example.set_index('SamplingOperations_code'),
        left_index=True,
        right_index=True,
        how='inner',
        suffixes=('_predicted', '_example')
    )

    # Extract the predicted status
    truesend = innerjoin["IBD_EQR_Status_Predicted"]

    # Rename and save to CSV
    truesend = truesend.rename("IBD_EQR_Status")
    truesend = truesend.to_frame()
    truesend.to_csv(output_file)
    print(f"Saved predictions to {output_file}")

    return truesend

In [34]:
df = to_status(con_nombre,ranges)

In [35]:
df

,SamplingOperations_code,IBD_pred,region,HERlvl1Code,HERlvl1Name,IBD_EQR_Status_Predicted
0,S02000010_20080811,15.062039,18,18,ALSACE,Good
1,S02000010_20100719,15.438521,18,18,ALSACE,Good
2,S02000010_20150811,13.819270,18,18,ALSACE,Moderate
3,S02000010_20160825,15.086803,18,18,ALSACE,Good
4,S02000010_20170703,16.036757,18,18,ALSACE,Good
...,...,...,...,...,...,...
5658,S06700075_20120611,20.008280,2,2,ALPES INTERNES,NaN
5659,S06700094_20210830,18.611227,2,2,ALPES INTERNES,Good
5660,S06700590_20210830,18.366359,2,2,ALPES INTERNES,Good
5661,S06710014_20210823,19.538874,2,2,ALPES INTERNES,High


In [36]:
results = df[['SamplingOperations_code','IBD_EQR_Status_Predicted']]

In [37]:
results

,SamplingOperations_code,IBD_EQR_Status_Predicted
0,S02000010_20080811,Good
1,S02000010_20100719,Good
2,S02000010_20150811,Moderate
3,S02000010_20160825,Good
4,S02000010_20170703,Good
...,...,...
5658,S06700075_20120611,NaN
5659,S06700094_20210830,Good
5660,S06700590_20210830,Good
5661,S06710014_20210823,High


In [38]:
results['IBD_EQR_Status_Predicted'] = results['IBD_EQR_Status_Predicted'].fillna('High')


C:\Users\narro\AppData\Local\Temp\ipykernel_14688\259161415.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results['IBD_EQR_Status_Predicted'] = results['IBD_EQR_Status_Predicted'].fillna('High')


In [40]:
results

,SamplingOperations_code,IBD_EQR_Status_Predicted
0,S02000010_20080811,Good
1,S02000010_20100719,Good
2,S02000010_20150811,Moderate
3,S02000010_20160825,Good
4,S02000010_20170703,Good
...,...,...
5658,S06700075_20120611,High
5659,S06700094_20210830,Good
5660,S06700590_20210830,Good
5661,S06710014_20210823,High


In [43]:
results = results.rename(columns={'IBD_EQR_Status_Predicted': 'IBD_EQR_Status'})


In [44]:
results.to_csv('cb_t.csv', index= False)